# Task 3: 检查 additional_benign 是否制造"捷径"或分布漂移

**目标**: 确认那 90 个来自另一张表的负样本没有让模型学到"识别数据来源"这种假信号。

- **实验 A**: 只保留正例 + 主表负例，排除 additional_benign 负例
- **实验 B (捷径探针)**: 训练分类器预测 `is_additional_benign`，如果 AUROC 很高 (>0.7)，说明特征能轻易区分数据来源 → 存在捷径风险

In [ ]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"

# ===== 加载数据 =====
df_main = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv")
df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")

# 对齐校验
assert len(df_main) == len(df_feat) == 2179
assert (df_main["Gene"].values == df_feat["Gene"].values).all()

# 把 source / AlphaMissense 从主表合并到特征表
df_feat["source"] = df_main["source"].values
df_feat["AlphaMissense_score"] = df_main["AlphaMissense score"].values

# 特征列定义
ID_COLS = ["KEY", "Gene", "reloc_v3"]
META_COLS = ["source", "AlphaMissense_score"]
NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
exclude_cols = ID_COLS + META_COLS + NAN_PLACEHOLDERS
feat_cols = [c for c in df_feat.columns if c not in exclude_cols]

X_full = df_feat[feat_cols].values.astype(np.float32)
y_5class = df_feat["reloc_v3"].values.astype(int)
y_bin = (y_5class > 0).astype(int)
groups = df_feat["Gene"].values
source_arr = df_feat["source"].values
am_score_arr = df_feat["AlphaMissense_score"].values.astype(np.float64)

n_total = len(y_bin)
n_pos = int(y_bin.sum())
n_neg = n_total - n_pos
base_rate = n_pos / n_total

print(f"数据加载完毕: n={n_total}, 正例={n_pos}, 负例={n_neg}, base_rate={base_rate:.4f}")
print(f"source 分布: {dict(zip(*np.unique(source_arr, return_counts=True)))}")
print(f"特征列数: {len(feat_cols)}")

# 统一 5 折 CV
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
print("统一 CV 已初始化 (StratifiedGroupKFold, n_splits=5, groups=Gene)")

# ===== CV 工具函数 =====
def cv_evaluate_binary(X, y, groups, sample_weight_mode="balanced"):
    """手动 StratifiedGroupKFold，每折内 Imputer→Scaler→XGBoost"""
    xgb_params = dict(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.5,
        objective="binary:logistic", eval_metric="aucpr",
        random_state=42, n_jobs=-1, tree_method="hist",
    )
    oof = np.zeros(len(y), dtype=np.float32)
    per_fold = []
    for fold, (tr_idx, te_idx) in enumerate(cv.split(X, y, groups)):
        X_tr_raw, X_te_raw = X[tr_idx], X[te_idx]
        y_tr = y[tr_idx]
        imp = SimpleImputer(strategy="median")
        scl = StandardScaler()
        X_tr = scl.fit_transform(imp.fit_transform(X_tr_raw)).astype(np.float32)
        X_te = scl.transform(imp.transform(X_te_raw)).astype(np.float32)
        sw = compute_sample_weight("balanced", y_tr)
        model = XGBClassifier(**xgb_params)
        model.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
        oof[te_idx] = model.predict_proba(X_te)[:, 1]
        y_te = y[te_idx]
        per_fold.append({
            "fold": fold,
            "auroc": roc_auc_score(y_te, oof[te_idx]),
            "auprc": average_precision_score(y_te, oof[te_idx]),
            "n": len(y_te), "n_pos": int(y_te.sum())
        })
    return oof, per_fold

def print_metrics(label, y_true, oof, per_fold=None):
    auroc = roc_auc_score(y_true, oof)
    auprc = average_precision_score(y_true, oof)
    n = len(y_true); n_pos = int(y_true.sum()); br = n_pos / n
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  n={n}, pos={n_pos}, base_rate={br:.4f}")
    print(f"  AUROC = {auroc:.4f}")
    print(f"  AUPRC = {auprc:.4f}  (base_rate={br:.4f})")
    if per_fold:
        fa = [f["auroc"] for f in per_fold]
        fp = [f["auprc"] for f in per_fold]
        print(f"  Per-fold AUROC: {[f'{v:.3f}' for v in fa]}  "
              f"mean={np.mean(fa):.4f} ± {np.std(fa):.4f}")
        print(f"  Per-fold AUPRC: {[f'{v:.3f}' for v in fp]}  "
              f"mean={np.mean(fp):.4f} ± {np.std(fp):.4f}")
    return {"label": label, "n": n, "n_pos": n_pos, "base_rate": br,
            "auroc": auroc, "auprc": auprc}

print("CV 工具函数就绪")


数据加载完毕: n=2179, 正例=222, 负例=1957, base_rate=0.1019
source 分布: {'additional_benign': np.int64(90), 'main': np.int64(2089)}
特征列数: 1288
统一 CV 已初始化 (StratifiedGroupKFold, n_splits=5, groups=Gene)
CV 工具函数就绪


## 3a. 实验 A: 排除 additional_benign 负例

In [2]:
# ===== 实验 A: 正例 + 主表负例 (排除 additional_benign) =====
mask_a = (source_arr == "main") | (y_bin == 1)
X_a = X_full[mask_a]
y_a = y_bin[mask_a]
g_a = groups[mask_a]

print(f"实验 A (正例+主表负例, 排除 additional_benign):")
print(f"  n={len(y_a)}, 正例={int(y_a.sum())}, base_rate={y_a.sum()/len(y_a):.4f}")
print(f"  排除的 additional_benign 负例: {(~mask_a).sum()} 行")

print("\n训练 v3 XGBoost ...")
oof_a, folds_a = cv_evaluate_binary(X_a, y_a, g_a)
r_a = print_metrics("实验 A: 正例+主表负例", y_a, oof_a, folds_a)

# 与全量对比
oof_full, _ = cv_evaluate_binary(X_full, y_bin, groups)
full_auroc = roc_auc_score(y_bin, oof_full)
print(f"\n  全量 AUROC={full_auroc:.4f}  vs  实验A AUROC={r_a['auroc']:.4f}  "
      f"(delta={r_a['auroc']-full_auroc:+.4f})")


实验 A (正例+主表负例, 排除 additional_benign):
  n=2089, 正例=222, base_rate=0.1063
  排除的 additional_benign 负例: 90 行

训练 v3 XGBoost ...

  实验 A: 正例+主表负例
  n=2089, pos=222, base_rate=0.1063
  AUROC = 0.5784
  AUPRC = 0.1495  (base_rate=0.1063)
  Per-fold AUROC: ['0.601', '0.603', '0.554', '0.562', '0.593']  mean=0.5826 ± 0.0207
  Per-fold AUPRC: ['0.169', '0.188', '0.129', '0.215', '0.158']  mean=0.1717 ± 0.0290

  全量 AUROC=0.5603  vs  实验A AUROC=0.5784  (delta=+0.0181)


## 3b. 实验 B: 捷径探针 —— 预测数据来源

构造标签 `is_additional_benign`（additional_benign=1，其余=0），用相同特征训练分类器。如果 AUROC 高，说明特征能区分数据来源。

In [3]:
# ===== 实验 B: 捷径探针 =====
y_probe = (source_arr == "additional_benign").astype(int)
n_probe_pos = int(y_probe.sum())
print(f"捷径探针标签: is_additional_benign=1 共 {n_probe_pos} 个 "
      f"(base_rate={n_probe_pos/len(y_probe):.4f})")

# 用 scale_pos_weight 处理极度不均衡
oof_probe, folds_probe = cv_evaluate_binary(
    X_full, y_probe, groups, sample_weight_mode="scale_pos_weight")

probe_auroc = roc_auc_score(y_probe, oof_probe)
probe_auprc = average_precision_score(y_probe, oof_probe)

print(f"\n捷径探针结果:")
print(f"  n={len(y_probe)}, positive={n_probe_pos}, base_rate={n_probe_pos/len(y_probe):.4f}")
print(f"  AUROC = {probe_auroc:.4f}")
print(f"  AUPRC = {probe_auprc:.4f} (base_rate={n_probe_pos/len(y_probe):.4f})")

print(f"\n{'='*60}")
if probe_auroc > 0.7:
    print(f"  ⚠️  警告: AUROC={probe_auroc:.3f} > 0.7")
    print(f"  特征能够区分数据来源 → 全量任务中存在被模型利用的捷径!")
    print(f"  建议: 在报告里标注此风险，或只用主表数据评估。")
else:
    print(f"  ✓  AUROC={probe_auroc:.3f} ≤ 0.7")
    print(f"  特征不能轻易区分数据来源 → 捷径风险低。")
print(f"{'='*60}")


捷径探针标签: is_additional_benign=1 共 90 个 (base_rate=0.0413)

捷径探针结果:
  n=2179, positive=90, base_rate=0.0413
  AUROC = 0.7432
  AUPRC = 0.1109 (base_rate=0.0413)

  ⚠️  警告: AUROC=0.743 > 0.7
  特征能够区分数据来源 → 全量任务中存在被模型利用的捷径!
  建议: 在报告里标注此风险，或只用主表数据评估。
